In [32]:
"""
Test the Linear Perturbation Prediction Method on Perturbation Map Data
======================================================================

This notebook adapts the bilinear ridge regression model from:
  "Deep learning-based predictions of gene perturbation effects 
   do not yet outperform simple linear baselines"

to the perturbation map spatial transcriptomics dataset.

Paper's model:  Y = A x K x B + center + baseline
  - Y: gene expression change matrix (genes x conditions)
  - A: gene embedding matrix (genes x dim)
  - B: perturbation embedding matrix (dim x conditions)
  - K: learned coefficient matrix (dim x dim)
  - center: row means of Y (centering)
  - baseline: control (None) mean expression

The ridge-regularized solution:
  K = (A'A + lI)^-1 A' Y B' (BB' + lI)^-1

Data: 0414/29198468/perturb_map_data_all.h5
  - X: 6363 cells x 1053 genes expression matrix
  - perturbation: 6 types (None=ctrl, KP, periphery, Tgfbr2, Ifngr2, Jak2)
  - tissue: normal (3747), tumor (2616)
  - batch: 4 slides (GSM5808054-8057)
  - pos: spatial coordinates

Evaluation uses evaluate() from evaluate.py (pertpy-based metrics).
"""

import h5py
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
import warnings, os, sys
import importlib
warnings.filterwarnings('ignore')

# Import the evaluate function from evaluate.py
sys.path.insert(0, 'd:/Codes/0513')
import evaluate as eval_mod
importlib.reload(eval_mod)  # Force reload to pick up edits to evaluate.py
from evaluate import evaluate

In [ ]:
# ============================================================================
# 1. Load Data
# ============================================================================
print("=" * 70)
print("STEP 1: Loading perturbation map data")
print("=" * 70)
#change to /mnt/172/wh/25-12/spatial/raw/pertubmap 
data_path = 'd:/Codes/0414/29198468/perturb_map_data_all.h5'
with h5py.File(data_path, 'r') as f:
    X = f['X'][:]           # (6363, 1053) expression matrix
    batch = f['batch'][:]   # (4, 6363) one-hot batch encoding
    gene_names = [g.decode() for g in f['gene'][:]]
    perturbation = [p.decode() for p in f['perturbation'][:]]
    pos = f['pos'][:]       # (2, 6363) spatial coordinates
    tissue = [t.decode() for t in f['tissue'][:]]

n_cells, n_genes = X.shape
print(f"  Cells: {n_cells}, Genes: {n_genes}")
print(f"  Density: {np.count_nonzero(X) / X.size:.2%}")

# Map batch one-hot to slide labels
slide_labels = ['GSM5808054', 'GSM5808055', 'GSM5808056', 'GSM5808057']
batch_idx = np.argmax(batch, axis=0)

STEP 1: Loading perturbation map data
  Cells: 6363, Genes: 1053
  Density: 37.63%


In [21]:
# ============================================================================
# 2. Pseudobulk: Aggregate cells by condition (perturbation type)
# ============================================================================
print("\n" + "=" * 70)
print("STEP 2: Pseudobulking - aggregating cells by perturbation type")
print("=" * 70)

pert_types = np.unique(perturbation)
print(f"  Perturbation types: {pert_types}")
print(f"  Cell counts per perturbation:")
for p in pert_types:
    count = sum(1 for x in perturbation if x == p)
    print(f"    {p}: {count}")

# Pseudobulk: sum expression across all cells of the same perturbation type
pseudobulk_expr = []
pseudobulk_labels = []
for p in pert_types:
    mask = np.array(perturbation) == p
    if mask.sum() > 0:
        pseudobulk_expr.append(X[mask].sum(axis=0))
        pseudobulk_labels.append(p)

pseudobulk_expr = np.array(pseudobulk_expr)  # (n_conditions, n_genes)
pseudobulk_labels = np.array(pseudobulk_labels)

# Transpose to match paper's convention: genes x conditions
Y_raw = pseudobulk_expr.T  # (n_genes, n_conditions)
print(f"  Pseudobulk matrix shape: {Y_raw.shape} (genes x conditions)")


STEP 2: Pseudobulking - aggregating cells by perturbation type
  Perturbation types: ['Ifngr2' 'Jak2' 'KP' 'None' 'Tgfbr2' 'periphery']
  Cell counts per perturbation:
    Ifngr2: 105
    Jak2: 21
    KP: 1061
    None: 4357
    Tgfbr2: 224
    periphery: 595
  Pseudobulk matrix shape: (1053, 6) (genes x conditions)


In [22]:
# ============================================================================
# 3. Compute Baseline (control mean) and Expression Changes
# ============================================================================
print("\n" + "=" * 70)
print("STEP 3: Computing baseline and expression changes")
print("=" * 70)

ctrl_idx = np.where(pseudobulk_labels == 'None')[0]
if len(ctrl_idx) == 0:
    raise ValueError("No 'None' control group found!")
print(f"  Control ('None') index: {ctrl_idx[0]}")

# Baseline = mean expression of control cells (per gene)
baseline = Y_raw[:, ctrl_idx[0]]  # (n_genes,)

# Expression change = pseudobulk expression - baseline
Y_change = Y_raw - baseline[:, np.newaxis]  # (n_genes, n_conditions)
print(f"  Expression change matrix shape: {Y_change.shape}")


STEP 3: Computing baseline and expression changes
  Control ('None') index: 3
  Expression change matrix shape: (1053, 6)


In [23]:
# ============================================================================
# 4. Get Gene and Perturbation Embeddings via PCA
# ============================================================================
print("\n" + "=" * 70)
print("STEP 4: Computing PCA embeddings for genes and perturbations")
print("=" * 70)

pca_dim = min(10, Y_change.shape[1] - 1, Y_change.shape[0] - 1)
print(f"  PCA dimension: {pca_dim}")

# Center the change matrix for PCA
Y_centered = Y_change - Y_change.mean(axis=1, keepdims=True)

# SVD on the change matrix
# Gene embedding (A): left singular vectors * singular values (genes x dim)
# Perturbation embedding (B): right singular vectors (dim x conditions)
U, s, Vt = np.linalg.svd(Y_centered, full_matrices=False)

gene_emb = U[:, :pca_dim] * s[:pca_dim]       # (n_genes, pca_dim)
pert_emb = Vt[:pca_dim, :]                     # (pca_dim, n_conditions)

print(f"  Gene embedding shape: {gene_emb.shape}")
print(f"  Perturbation embedding shape: {pert_emb.shape}")


STEP 4: Computing PCA embeddings for genes and perturbations
  PCA dimension: 5
  Gene embedding shape: (1053, 5)
  Perturbation embedding shape: (5, 6)


In [24]:
# ============================================================================
# 5. Solve Bilinear Ridge Regression: Y = A x K x B
# ============================================================================
print("\n" + "=" * 70)
print("STEP 5: Solving bilinear ridge regression")
print("=" * 70)

def solve_y_axb(Y, A, B, ridge_penalty=0.1):
    """
    Solve Y = A @ K @ B with ridge regularization.
    
    Closed-form solution:
    K = (A'A + lI)^-1 A' Y B' (BB' + lI)^-1
    """
    center = Y.mean(axis=1)
    Y_c = Y - center[:, np.newaxis]
    
    n_dim = A.shape[1]
    
    # (A'A + lI)^-1 A'
    ATA = A.T @ A
    ATA_reg = ATA + np.eye(n_dim) * ridge_penalty
    ATA_inv = np.linalg.solve(ATA_reg, np.eye(n_dim))
    left = ATA_inv @ A.T  # (dim, n_genes)
    
    # Y B' (BB' + lI)^-1
    BBT = B @ B.T
    BBT_reg = BBT + np.eye(n_dim) * ridge_penalty
    BBT_inv = np.linalg.solve(BBT_reg, np.eye(n_dim))
    right = Y_c @ B.T @ BBT_inv  # (n_genes, dim)
    
    # K = left @ right
    K = left @ right  # (dim, dim)
    
    K[np.isnan(K)] = 0
    return K, center

ridge_penalty = 0.1
K, center = solve_y_axb(Y_change, gene_emb, pert_emb, ridge_penalty=ridge_penalty)
print(f"  Coefficient matrix K shape: {K.shape}")
print(f"  Ridge penalty l = {ridge_penalty}")


STEP 5: Solving bilinear ridge regression
  Coefficient matrix K shape: (5, 5)
  Ridge penalty l = 0.1


In [25]:
print(K)

[[ 9.09090909e-01 -6.93889390e-17  8.32667268e-17  4.81385765e-17
   1.35877637e-16]
 [ 2.22044605e-16  9.09090909e-01 -2.22044605e-16 -3.12250226e-17
   1.17093835e-17]
 [ 1.77635684e-15 -6.66133815e-16  9.09090909e-01 -1.59594560e-16
   4.42354486e-17]
 [ 1.77635684e-15 -5.55111512e-16 -1.11022302e-16  9.09090909e-01
   1.15055535e-15]
 [ 0.00000000e+00 -8.88178420e-16 -2.22044605e-15 -4.85722573e-17
   9.09090906e-01]]


In [26]:
# ============================================================================
# 6. Make Predictions (per-condition pseudobulk level)
# ============================================================================
print("\n" + "=" * 70)
print("STEP 6: Making predictions")
print("=" * 70)

# Prediction: Y_pred = A @ K @ B + center + baseline
Y_pred_change = gene_emb @ K @ pert_emb + center[:, np.newaxis]
Y_pred = Y_pred_change + baseline[:, np.newaxis]

print(f"  Predicted expression shape: {Y_pred.shape}")
print(f"  Conditions: {pseudobulk_labels}")


STEP 6: Making predictions
  Predicted expression shape: (1053, 6)
  Conditions: ['Ifngr2' 'Jak2' 'KP' 'None' 'Tgfbr2' 'periphery']


In [33]:
# ============================================================================
# 7. Evaluate using evaluate() from evaluate.py
# ============================================================================
print("\n" + "=" * 70)
print("STEP 7: Evaluating predictions using pertpy-based metrics")
print("=" * 70)

# The evaluate() function expects:
#   pred_expr:  (N_pred, G) numpy array - predicted expression for each cell
#   true_expr:  AnnData (N_pred, G) with .layers['logNor'] - real stimulated cells
#   ctrl_expr:  (N_ctrl, G) numpy array - real control cells
#   gene_names: list of gene names

all_results = []

for i, cond in enumerate(pseudobulk_labels):
    if cond == 'None':
        continue  # Skip control
    
    print(f"\n  {'='*50}")
    print(f"  Evaluating: {cond}")
    print(f"  {'='*50}")
    
    # Get cells of this perturbation type
    cond_mask = np.array(perturbation) == cond
    n_cond_cells = cond_mask.sum()
    print(f"  Number of cells: {n_cond_cells}")
    
    # Get control cells (all 'None' cells)
    ctrl_mask = np.array(perturbation) == 'None'
    ctrl_expr = X[ctrl_mask]  # (N_ctrl, G)
    
    # Get real stimulated cells for this condition
    stim_expr = X[cond_mask]  # (N_cond, G)
    
    # Create AnnData for true expression with logNor layer
    true_adata = sc.AnnData(
        X=sparse.csr_matrix(stim_expr),
        obs=pd.DataFrame({'perturbation': [cond] * n_cond_cells}),
        var=pd.DataFrame(index=gene_names)
    )
    # Compute log-normalized layer
    true_adata.layers['logNor'] = stim_expr.copy()
    sc.pp.normalize_total(true_adata, target_sum=1e4)
    sc.pp.log1p(true_adata)
    true_adata.layers['logNor'] = true_adata.X.toarray() if sparse.issparse(true_adata.X) else true_adata.X.copy()
    # Restore raw counts in X
    true_adata.X = sparse.csr_matrix(stim_expr)
    
    # Generate per-cell predictions: each cell gets the pseudobulk-level prediction
    pred_expr = np.tile(Y_pred[:, i], (n_cond_cells, 1))  # (N_cond, G)
    
    print(f"  pred_expr shape: {pred_expr.shape}")
    print(f"  true_expr shape: {true_adata.shape}")
    print(f"  ctrl_expr shape: {ctrl_expr.shape}")
    
    # Call evaluate()
    save_dir = f'd:/Codes/0513/evaluation_results/{cond}'
    try:
        result_df = evaluate(
            pred_expr=pred_expr,
            true_expr=true_adata,
            ctrl_expr=ctrl_expr,
            gene_names=gene_names,
            condition_col='perturbation',
            save_dir=save_dir,
            top_n=100,
            de_method='wilcoxon',
            subsample_n=2000
        )
        result_df.insert(0, 'condition', cond)
        all_results.append(result_df)
    except Exception as e:
        print(f"  [ERROR] evaluate() failed for {cond}: {e}")

# Combine all results
if all_results:
    combined_results = pd.concat(all_results, ignore_index=True)
    print("\n" + "=" * 70)
    print("COMBINED EVALUATION RESULTS")
    print("=" * 70)
    print(combined_results.round(4).to_string(index=False))
    
    # Save combined results
    os.makedirs('d:/Codes/0513/evaluation_results', exist_ok=True)
    combined_results.to_csv('d:/Codes/0513/evaluation_results/all_conditions_metrics.csv', index=False)
    print("\nSaved combined results to: d:/Codes/0513/evaluation_results/all_conditions_metrics.csv")
else:
    print("No results generated.")


STEP 7: Evaluating predictions using pertpy-based metrics

  Evaluating: Ifngr2
  Number of cells: 105
  pred_expr shape: (105, 1053)
  true_expr shape: (105, 1053)
  ctrl_expr shape: (4357, 1053)
Shapes for vstack -> Ctrl:(4357, 1053), Stim:(105, 1053), Pred:(105, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/Ifngr2
   mse_all_Ifngr2  pearson_distance_all_Ifngr2  edistance_all_Ifngr2  \
0          0.3681                       0.1174               19.6881   

   wasserstein_all_Ifngr2  sym_kldiv_all_Ifngr2  mse_all  \
0                 19.6881          8.284608e+15   0.3681   

   pearson_distance_all  edistance_all  wasserstein_all  sym_kldiv_all  \
0                0.1174        19.6881          19.6881   8.284608e+15   

   mse_top100_Ifngr2  pearson_distance_top100_Ifngr2  edistance_top100_Ifngr2  \
0             0.2759                          0.0715                   9.1249   

   wasserstein_top100_Ifngr2  sym_kldiv_top100_Ifngr2  \
0                   155.0522             7.752607e+15   

   common_degs_top100_Ifngr2  
0                         42  

  Evaluating: Jak2
  Number of cells: 21
  pred_expr shape: (21, 1053)
  true_expr shape: (21, 1053)
  ctrl_expr shape: (4357, 1053)
Shapes for vstack -> Ctrl:(4357, 1053), Stim:(21, 

Saved evaluation at: d:/Codes/0513/evaluation_results/Jak2
   mse_all_Jak2  pearson_distance_all_Jak2  edistance_all_Jak2  \
0        0.6876                     0.2134             26.9078   

   wasserstein_all_Jak2  sym_kldiv_all_Jak2  mse_all  pearson_distance_all  \
0               26.9078        8.733449e+15   0.6876                0.2134   

   edistance_all  wasserstein_all  sym_kldiv_all  mse_top100_Jak2  \
0        26.9078          26.9078   8.733449e+15           1.7898   

   pearson_distance_top100_Jak2  edistance_top100_Jak2  \
0                        0.2087                21.1005   

   wasserstein_top100_Jak2  sym_kldiv_top100_Jak2  common_degs_top100_Jak2  
0                 219.3459           1.096729e+16                       30  

  Evaluating: KP
  Number of cells: 1061
  pred_expr shape: (1061, 1053)
  true_expr shape: (1061, 1053)
  ctrl_expr shape: (4357, 1053)
Shapes for vstack -> Ctrl:(4357, 1053), Stim:(1061, 1053), Pred:(1061, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/KP
   mse_all_KP  pearson_distance_all_KP  edistance_all_KP  wasserstein_all_KP  \
0      0.0938                   0.0313            9.9375              9.9375   

   sym_kldiv_all_KP  mse_all  pearson_distance_all  edistance_all  \
0      7.370048e+15   0.0938                0.0313         9.9375   

   wasserstein_all  sym_kldiv_all  mse_top100_KP  pearson_distance_top100_KP  \
0           9.9375   7.370048e+15         0.0856                      0.0536   

   edistance_top100_KP  wasserstein_top100_KP  sym_kldiv_top100_KP  \
0               7.2795               132.9773         6.648866e+15   

   common_degs_top100_KP  
0                     38  

  Evaluating: Tgfbr2
  Number of cells: 224
  pred_expr shape: (224, 1053)
  true_expr shape: (224, 1053)
  ctrl_expr shape: (4357, 1053)
Shapes for vstack -> Ctrl:(4357, 1053), Stim:(224, 1053), Pred:(224, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/Tgfbr2
   mse_all_Tgfbr2  pearson_distance_all_Tgfbr2  edistance_all_Tgfbr2  \
0          0.2491                       0.0715               16.1972   

   wasserstein_all_Tgfbr2  sym_kldiv_all_Tgfbr2  mse_all  \
0                 16.1972          7.508777e+15   0.2491   

   pearson_distance_all  edistance_all  wasserstein_all  sym_kldiv_all  \
0                0.0715        16.1972          16.1972   7.508777e+15   

   mse_top100_Tgfbr2  pearson_distance_top100_Tgfbr2  edistance_top100_Tgfbr2  \
0             0.2084                          0.0858                   8.0832   

   wasserstein_top100_Tgfbr2  sym_kldiv_top100_Tgfbr2  \
0                   130.6621             6.533103e+15   

   common_degs_top100_Tgfbr2  
0                         50  

  Evaluating: periphery
  Number of cells: 595
  pred_expr shape: (595, 1053)
  true_expr shape: (595, 1053)
  ctrl_expr shape: (4357, 1053)
Shapes for vstack -> Ctrl:(4357, 1053), St

Saved evaluation at: d:/Codes/0513/evaluation_results/periphery
   mse_all_periphery  pearson_distance_all_periphery  edistance_all_periphery  \
0             0.0767                          0.0355                   8.9865   

   wasserstein_all_periphery  sym_kldiv_all_periphery  mse_all  \
0                     8.9865             7.671845e+15   0.0767   

   pearson_distance_all  edistance_all  wasserstein_all  sym_kldiv_all  \
0                0.0355         8.9865           8.9865   7.671845e+15   

   mse_top100_periphery  pearson_distance_top100_periphery  \
0                 0.058                             0.0408   

   edistance_top100_periphery  wasserstein_top100_periphery  \
0                      6.4954                      110.5614   

   sym_kldiv_top100_periphery  common_degs_top100_periphery  
0                5.528070e+15                             1  

COMBINED EVALUATION RESULTS
condition  mse_all_Ifngr2  pearson_distance_all_Ifngr2  edistance_all_Ifngr2  wasserst

In [34]:
# ============================================================================
# 8. Compare with Simple Baselines using evaluate()
# ============================================================================
print("\n" + "=" * 70)
print("STEP 8: Comparing with simple baselines using evaluate()")
print("=" * 70)

def evaluate_baseline(baseline_name, pred_expr_fn):
    """Evaluate a baseline prediction method."""
    all_results = []
    
    for cond in pert_types:
        if cond == 'None':
            continue
        
        print(f"\n  --- {baseline_name}: {cond} ---")
        
        cond_mask = np.array(perturbation) == cond
        n_cond_cells = cond_mask.sum()
        ctrl_mask = np.array(perturbation) == 'None'
        ctrl_expr = X[ctrl_mask]
        stim_expr = X[cond_mask]
        
        # Create AnnData for true expression
        true_adata = sc.AnnData(
            X=sparse.csr_matrix(stim_expr),
            obs=pd.DataFrame({'perturbation': [cond] * n_cond_cells}),
            var=pd.DataFrame(index=gene_names)
        )
        true_adata.layers['logNor'] = stim_expr.copy()
        sc.pp.normalize_total(true_adata, target_sum=1e4)
        sc.pp.log1p(true_adata)
        true_adata.layers['logNor'] = true_adata.X.toarray() if sparse.issparse(true_adata.X) else true_adata.X.copy()
        true_adata.X = sparse.csr_matrix(stim_expr)
        
        # Generate baseline predictions
        pred_expr = pred_expr_fn(cond, n_cond_cells, stim_expr, ctrl_expr)
        
        save_dir = f'd:/Codes/0513/evaluation_results/{baseline_name}/{cond}'
        try:
            result_df = evaluate(
                pred_expr=pred_expr,
                true_expr=true_adata,
                ctrl_expr=ctrl_expr,
                gene_names=gene_names,
                condition_col='perturbation',
                save_dir=save_dir,
                top_n=100,
                de_method='wilcoxon',
                subsample_n=2000
            )
            result_df.insert(0, 'condition', cond)
            result_df.insert(0, 'method', baseline_name)
            all_results.append(result_df)
        except Exception as e:
            print(f"    [ERROR] {e}")
    
    return pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

# Baseline 1: Mean prediction (predict mean expression across all training conditions)
print("\n" + "-" * 50)
print("Baseline 1: Mean prediction")
print("-" * 50)

# Mean across all non-ctrl conditions at pseudobulk level
non_ctrl_idx = [i for i, p in enumerate(pseudobulk_labels) if p != 'None']
mean_pseudobulk = Y_raw[:, non_ctrl_idx].mean(axis=1)  # (n_genes,)

def mean_baseline_pred(cond, n_cells, stim_expr, ctrl_expr):
    return np.tile(mean_pseudobulk, (n_cells, 1))

mean_results = evaluate_baseline('mean_baseline', mean_baseline_pred)

# Baseline 2: Zero prediction (predict no change = control expression)
print("\n" + "-" * 50)
print("Baseline 2: Zero/control prediction")
print("-" * 50)

# Use the mean control expression
ctrl_mean_expr = X[np.array(perturbation) == 'None'].mean(axis=0)  # (n_genes,)

def zero_baseline_pred(cond, n_cells, stim_expr, ctrl_expr):
    return np.tile(ctrl_mean_expr, (n_cells, 1))

zero_results = evaluate_baseline('zero_baseline', zero_baseline_pred)

# Combine all baseline results
all_baseline_results = pd.concat(
    [df for df in [mean_results, zero_results] if not df.empty],
    ignore_index=True
)

if not all_baseline_results.empty:
    print("\n" + "=" * 70)
    print("BASELINE COMPARISON RESULTS")
    print("=" * 70)
    print(all_baseline_results.round(4).to_string(index=False))
    all_baseline_results.to_csv('d:/Codes/0513/evaluation_results/baseline_comparison.csv', index=False)


STEP 8: Comparing with simple baselines using evaluate()

--------------------------------------------------
Baseline 1: Mean prediction
--------------------------------------------------

  --- mean_baseline: Ifngr2 ---
Shapes for vstack -> Ctrl:(4357, 1053), Stim:(105, 1053), Pred:(105, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/mean_baseline/Ifngr2
   mse_all_Ifngr2  pearson_distance_all_Ifngr2  edistance_all_Ifngr2  \
0          0.4311                        0.149               21.3053   

   wasserstein_all_Ifngr2  sym_kldiv_all_Ifngr2  mse_all  \
0                 21.3053          8.598068e+15   0.4311   

   pearson_distance_all  edistance_all  wasserstein_all  sym_kldiv_all  \
0                 0.149        21.3053          21.3053   8.598068e+15   

   mse_top100_Ifngr2  pearson_distance_top100_Ifngr2  edistance_top100_Ifngr2  \
0             0.4554                          0.0928                  10.5725   

   wasserstein_top100_Ifngr2  sym_kldiv_top100_Ifngr2  \
0                   172.9995             8.649973e+15   

   common_degs_top100_Ifngr2  
0                         30  

  --- mean_baseline: Jak2 ---
Shapes for vstack -> Ctrl:(4357, 1053), Stim:(21, 1053), Pred:(21, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/mean_baseline/Jak2
   mse_all_Jak2  pearson_distance_all_Jak2  edistance_all_Jak2  \
0        0.5589                     0.1777             24.2594   

   wasserstein_all_Jak2  sym_kldiv_all_Jak2  mse_all  pearson_distance_all  \
0               24.2594        7.997319e+15   0.5589                0.1777   

   edistance_all  wasserstein_all  sym_kldiv_all  mse_top100_Jak2  \
0        24.2594          24.2594   7.997319e+15           1.9033   

   pearson_distance_top100_Jak2  edistance_top100_Jak2  \
0                        0.3909                21.8668   

   wasserstein_top100_Jak2  sym_kldiv_top100_Jak2  common_degs_top100_Jak2  
0                 230.6996           1.153498e+16                       36  

  --- mean_baseline: KP ---
Shapes for vstack -> Ctrl:(4357, 1053), Stim:(1061, 1053), Pred:(1061, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/mean_baseline/KP
   mse_all_KP  pearson_distance_all_KP  edistance_all_KP  wasserstein_all_KP  \
0      0.1326                   0.0392           11.8167             11.8167   

   sym_kldiv_all_KP  mse_all  pearson_distance_all  edistance_all  \
0      7.564163e+15   0.1326                0.0392        11.8167   

   wasserstein_all  sym_kldiv_all  mse_top100_KP  pearson_distance_top100_KP  \
0          11.8167   7.564163e+15         0.1111                      0.0517   

   edistance_top100_KP  wasserstein_top100_KP  sym_kldiv_top100_KP  \
0                7.531               135.5272         6.776357e+15   

   common_degs_top100_KP  
0                     29  

  --- mean_baseline: Tgfbr2 ---
Shapes for vstack -> Ctrl:(4357, 1053), Stim:(224, 1053), Pred:(224, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/mean_baseline/Tgfbr2
   mse_all_Tgfbr2  pearson_distance_all_Tgfbr2  edistance_all_Tgfbr2  \
0          0.3888                       0.1262               20.2342   

   wasserstein_all_Tgfbr2  sym_kldiv_all_Tgfbr2  mse_all  \
0                 20.2342          8.206735e+15   0.3888   

   pearson_distance_all  edistance_all  wasserstein_all  sym_kldiv_all  \
0                0.1262        20.2342          20.2342   8.206735e+15   

   mse_top100_Tgfbr2  pearson_distance_top100_Tgfbr2  edistance_top100_Tgfbr2  \
0              0.475                          0.1486                  10.3702   

   wasserstein_top100_Tgfbr2  sym_kldiv_top100_Tgfbr2  \
0                   157.3228             7.866141e+15   

   common_degs_top100_Tgfbr2  
0                         30  

  --- mean_baseline: periphery ---
Shapes for vstack -> Ctrl:(4357, 1053), Stim:(595, 1053), Pred:(595, 1053)


...

Saved evaluation at: d:/Codes/0513/evaluation_results/zero_baseline/Ifngr2
   mse_all_Ifngr2  pearson_distance_all_Ifngr2  edistance_all_Ifngr2  \
0          0.8826                       0.3482               30.4864   

   wasserstein_all_Ifngr2  sym_kldiv_all_Ifngr2  mse_all  \
0                 30.4864          1.087094e+16   0.8826   

   pearson_distance_all  edistance_all  wasserstein_all  sym_kldiv_all  \
0                0.3482        30.4864          30.4864   1.087094e+16   

   mse_top100_Ifngr2  pearson_distance_top100_Ifngr2  edistance_top100_Ifngr2  \
0             1.9185                          0.1199                  19.8817   

   wasserstein_top100_Ifngr2  sym_kldiv_top100_Ifngr2  \
0                   319.3089             1.596544e+16   

   common_degs_top100_Ifngr2  
0                          5  

  --- zero_baseline: Jak2 ---
Shapes for vstack -> Ctrl:(4357, 1053), Stim:(21, 1053), Pred:(21, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/zero_baseline/Jak2
   mse_all_Jak2  pearson_distance_all_Jak2  edistance_all_Jak2  \
0         1.105                       0.39             34.1112   

   wasserstein_all_Jak2  sym_kldiv_all_Jak2  mse_all  pearson_distance_all  \
0               34.1112        1.093979e+16    1.105                  0.39   

   edistance_all  wasserstein_all  sym_kldiv_all  mse_top100_Jak2  \
0        34.1112          34.1112   1.093979e+16           4.3123   

   pearson_distance_top100_Jak2  edistance_top100_Jak2  \
0                        0.5001                34.9239   

   wasserstein_top100_Jak2  sym_kldiv_top100_Jak2  common_degs_top100_Jak2  
0                 471.5972           2.357986e+16                       15  

  --- zero_baseline: KP ---
Shapes for vstack -> Ctrl:(4357, 1053), Stim:(1061, 1053), Pred:(1061, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/zero_baseline/KP
   mse_all_KP  pearson_distance_all_KP  edistance_all_KP  wasserstein_all_KP  \
0      0.4608                   0.1854           22.0288             22.0288   

   sym_kldiv_all_KP  mse_all  pearson_distance_all  edistance_all  \
0      9.205345e+15   0.4608                0.1854        22.0288   

   wasserstein_all  sym_kldiv_all  mse_top100_KP  pearson_distance_top100_KP  \
0          22.0288   9.205345e+15         1.1405                       0.106   

   edistance_top100_KP  wasserstein_top100_KP  sym_kldiv_top100_KP  \
0              15.0863               238.4713         1.192356e+16   

   common_degs_top100_KP  
0                      0  

  --- zero_baseline: Tgfbr2 ---
Shapes for vstack -> Ctrl:(4357, 1053), Stim:(224, 1053), Pred:(224, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/zero_baseline/Tgfbr2
   mse_all_Tgfbr2  pearson_distance_all_Tgfbr2  edistance_all_Tgfbr2  \
0          0.8689                        0.332                30.248   

   wasserstein_all_Tgfbr2  sym_kldiv_all_Tgfbr2  mse_all  \
0                  30.248          1.061642e+16   0.8689   

   pearson_distance_all  edistance_all  wasserstein_all  sym_kldiv_all  \
0                 0.332         30.248           30.248   1.061642e+16   

   mse_top100_Tgfbr2  pearson_distance_top100_Tgfbr2  edistance_top100_Tgfbr2  \
0             1.8892                          0.2029                  19.8234   

   wasserstein_top100_Tgfbr2  sym_kldiv_top100_Tgfbr2  \
0                   298.7432             1.493716e+16   

   common_degs_top100_Tgfbr2  
0                          2  

  --- zero_baseline: periphery ---
Shapes for vstack -> Ctrl:(4357, 1053), Stim:(595, 1053), Pred:(595, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/zero_baseline/periphery
   mse_all_periphery  pearson_distance_all_periphery  edistance_all_periphery  \
0             0.1254                          0.0655                  11.4912   

   wasserstein_all_periphery  sym_kldiv_all_periphery  mse_all  \
0                    11.4912             7.915383e+15   0.1254   

   pearson_distance_all  edistance_all  wasserstein_all  sym_kldiv_all  \
0                0.0655        11.4912          11.4912   7.915383e+15   

   mse_top100_periphery  pearson_distance_top100_periphery  \
0                0.2507                             0.0846   

   edistance_top100_periphery  wasserstein_top100_periphery  \
0                      8.3433                      129.8286   

   sym_kldiv_top100_periphery  common_degs_top100_periphery  
0                6.491427e+15                             0  

BASELINE COMPARISON RESULTS
       method condition  mse_all_Ifngr2  pearson_distance_all_Ifngr2  ed

In [35]:
# ============================================================================
# 9. Per-Slide Analysis (leave-one-slide-out cross-validation)
# ============================================================================
print("\n" + "=" * 70)
print("STEP 9: Per-slide analysis (leave-one-slide-out cross-validation)")
print("=" * 70)

cv_results = []

for held_out_slide_idx, held_out_slide in enumerate(slide_labels):
    print(f"\n  --- Held-out slide: {held_out_slide} ---")
    
    # Split cells: train on other slides, test on held-out slide
    train_mask = batch_idx != held_out_slide_idx
    test_mask = batch_idx == held_out_slide_idx
    
    # Pseudobulk training conditions
    train_expr_list = []
    train_labels_list = []
    
    for p in pert_types:
        train_cells = (np.array(perturbation) == p) & train_mask
        if train_cells.sum() > 0:
            train_expr_list.append(X[train_cells].sum(axis=0))
            train_labels_list.append(p)
    
    train_expr_arr = np.array(train_expr_list).T  # (n_genes, n_train_conditions)
    train_labels_arr = np.array(train_labels_list)
    
    # Baseline from training control
    train_ctrl_idx = np.where(train_labels_arr == 'None')[0]
    if len(train_ctrl_idx) == 0:
        print(f"    Skipping: no control in training set")
        continue
    train_baseline = train_expr_arr[:, train_ctrl_idx[0]]
    
    # Training change matrix
    train_change = train_expr_arr - train_baseline[:, np.newaxis]
    
    # PCA on training data
    pca_dim_cv = min(10, train_change.shape[1] - 1, train_change.shape[0] - 1)
    if pca_dim_cv < 2:
        print(f"    Skipping: insufficient dimensions for PCA (dim={pca_dim_cv})")
        continue
    
    train_centered = train_change - train_change.mean(axis=1, keepdims=True)
    U_cv, s_cv, Vt_cv = np.linalg.svd(train_centered, full_matrices=False)
    
    gene_emb_cv = U_cv[:, :pca_dim_cv] * s_cv[:pca_dim_cv]
    pert_emb_cv = Vt_cv[:pca_dim_cv, :]
    
    # Solve ridge regression on training data
    K_cv, center_cv = solve_y_axb(train_change, gene_emb_cv, pert_emb_cv, ridge_penalty=ridge_penalty)
    
    # Predict on held-out slide cells
    for p in pert_types:
        if p == 'None':
            continue
        
        test_cells_mask = (np.array(perturbation) == p) & test_mask
        n_test_cells = test_cells_mask.sum()
        if n_test_cells == 0:
            continue
        
        # Get perturbation embedding for this condition
        if p in train_labels_arr:
            train_j = np.where(train_labels_arr == p)[0][0]
            p_emb = pert_emb_cv[:, train_j:train_j+1]  # (dim, 1)
        else:
            p_emb = np.zeros((pca_dim_cv, 1))
        
        # Predict pseudobulk-level expression
        pred_change = gene_emb_cv @ K_cv @ p_emb + center_cv[:, np.newaxis]
        pred_expr_cv = pred_change + train_baseline[:, np.newaxis]
        
        # Tile to per-cell predictions
        pred_expr_per_cell = np.tile(pred_expr_cv.T, (n_test_cells, 1))
        
        # Get real expression
        stim_expr_cv = X[test_cells_mask]
        ctrl_expr_cv = X[(np.array(perturbation) == 'None') & train_mask]
        
        # Create AnnData
        true_adata_cv = sc.AnnData(
            X=sparse.csr_matrix(stim_expr_cv),
            obs=pd.DataFrame({'perturbation': [p] * n_test_cells}),
            var=pd.DataFrame(index=gene_names)
        )
        true_adata_cv.layers['logNor'] = stim_expr_cv.copy()
        sc.pp.normalize_total(true_adata_cv, target_sum=1e4)
        sc.pp.log1p(true_adata_cv)
        true_adata_cv.layers['logNor'] = true_adata_cv.X.toarray() if sparse.issparse(true_adata_cv.X) else true_adata_cv.X.copy()
        true_adata_cv.X = sparse.csr_matrix(stim_expr_cv)
        
        # Evaluate
        save_dir_cv = f'd:/Codes/0513/evaluation_results/cv/{held_out_slide}/{p}'
        try:
            result_cv = evaluate(
                pred_expr=pred_expr_per_cell,
                true_expr=true_adata_cv,
                ctrl_expr=ctrl_expr_cv,
                gene_names=gene_names,
                condition_col='perturbation',
                save_dir=save_dir_cv,
                top_n=100,
                de_method='wilcoxon',
                subsample_n=2000
            )
            result_cv.insert(0, 'condition', p)
            result_cv.insert(0, 'held_out_slide', held_out_slide)
            cv_results.append(result_cv)
        except Exception as e:
            print(f"    [ERROR] {e}")

if cv_results:
    cv_combined = pd.concat(cv_results, ignore_index=True)
    print("\n" + "=" * 70)
    print("CROSS-VALIDATION RESULTS")
    print("=" * 70)
    print(cv_combined.round(4).to_string(index=False))
    cv_combined.to_csv('d:/Codes/0513/evaluation_results/cross_validation_metrics.csv', index=False)

print("\n" + "=" * 70)
print("DONE! Linear perturbation prediction test complete.")
print("=" * 70)


STEP 9: Per-slide analysis (leave-one-slide-out cross-validation)

  --- Held-out slide: GSM5808054 ---
Shapes for vstack -> Ctrl:(2932, 1053), Stim:(21, 1053), Pred:(21, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/cv/GSM5808054/Jak2
   mse_all_Jak2  pearson_distance_all_Jak2  edistance_all_Jak2  \
0        0.8033                     0.2745             29.0835   

   wasserstein_all_Jak2  sym_kldiv_all_Jak2  mse_all  pearson_distance_all  \
0               29.0835        9.325950e+15   0.8033                0.2745   

   edistance_all  wasserstein_all  sym_kldiv_all  mse_top100_Jak2  \
0        29.0835          29.0835   9.325950e+15           3.0949   

   pearson_distance_top100_Jak2  edistance_top100_Jak2  \
0                        0.4679                28.5797   

   wasserstein_top100_Jak2  sym_kldiv_top100_Jak2  common_degs_top100_Jak2  
0                 354.9579           1.774789e+16                       13  
Shapes for vstack -> Ctrl:(2932, 1053), Stim:(233, 1053), Pred:(233, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/cv/GSM5808054/KP
   mse_all_KP  pearson_distance_all_KP  edistance_all_KP  wasserstein_all_KP  \
0      0.1277                   0.0457           11.5976             11.5976   

   sym_kldiv_all_KP  mse_all  pearson_distance_all  edistance_all  \
0      7.699722e+15   0.1277                0.0457        11.5976   

   wasserstein_all  sym_kldiv_all  mse_top100_KP  pearson_distance_top100_KP  \
0          11.5976   7.699722e+15         0.1219                       0.116   

   edistance_top100_KP  wasserstein_top100_KP  sym_kldiv_top100_KP  \
0               7.7728                144.802         7.240099e+15   

   common_degs_top100_KP  
0                     22  
Shapes for vstack -> Ctrl:(2932, 1053), Stim:(75, 1053), Pred:(75, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/cv/GSM5808054/Tgfbr2
   mse_all_Tgfbr2  pearson_distance_all_Tgfbr2  edistance_all_Tgfbr2  \
0          0.4639                       0.1225               22.1014   

   wasserstein_all_Tgfbr2  sym_kldiv_all_Tgfbr2  mse_all  \
0                 22.1014          7.600846e+15   0.4639   

   pearson_distance_all  edistance_all  wasserstein_all  sym_kldiv_all  \
0                0.1225        22.1014          22.1014   7.600846e+15   

   mse_top100_Tgfbr2  pearson_distance_top100_Tgfbr2  edistance_top100_Tgfbr2  \
0             0.6071                          0.1665                  11.3234   

   wasserstein_top100_Tgfbr2  sym_kldiv_top100_Tgfbr2  \
0                   150.2967             7.514835e+15   

   common_degs_top100_Tgfbr2  
0                         41  
Shapes for vstack -> Ctrl:(2932, 1053), Stim:(149, 1053), Pred:(149, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/cv/GSM5808054/periphery
   mse_all_periphery  pearson_distance_all_periphery  edistance_all_periphery  \
0             0.1526                          0.0746                  12.6756   

   wasserstein_all_periphery  sym_kldiv_all_periphery  mse_all  \
0                    12.6756             8.048501e+15   0.1526   

   pearson_distance_all  edistance_all  wasserstein_all  sym_kldiv_all  \
0                0.0746        12.6756          12.6756   8.048501e+15   

   mse_top100_periphery  pearson_distance_top100_periphery  \
0                0.2873                             0.1494   

   edistance_top100_periphery  wasserstein_top100_periphery  \
0                      8.7127                      132.1846   

   sym_kldiv_top100_periphery  common_degs_top100_periphery  
0                6.609229e+15                             1  

  --- Held-out slide: GSM5808055 ---
Shapes for vstack -> Ctrl:(3062, 1053), Stim:(41, 1053), Pred:(

...

Saved evaluation at: d:/Codes/0513/evaluation_results/cv/GSM5808056/KP
   mse_all_KP  pearson_distance_all_KP  edistance_all_KP  wasserstein_all_KP  \
0       0.113                   0.0434           10.9084             10.9084   

   sym_kldiv_all_KP  mse_all  pearson_distance_all  edistance_all  \
0      7.389652e+15    0.113                0.0434        10.9084   

   wasserstein_all  sym_kldiv_all  mse_top100_KP  pearson_distance_top100_KP  \
0          10.9084   7.389652e+15         0.1271                      0.0922   

   edistance_top100_KP  wasserstein_top100_KP  sym_kldiv_top100_KP  \
0               7.3765               124.9097         6.245486e+15   

   common_degs_top100_KP  
0                     34  
Shapes for vstack -> Ctrl:(3622, 1053), Stim:(148, 1053), Pred:(148, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/cv/GSM5808056/periphery
   mse_all_periphery  pearson_distance_all_periphery  edistance_all_periphery  \
0             0.1504                           0.075                  12.5831   

   wasserstein_all_periphery  sym_kldiv_all_periphery  mse_all  \
0                    12.5831             7.647484e+15   0.1504   

   pearson_distance_all  edistance_all  wasserstein_all  sym_kldiv_all  \
0                 0.075        12.5831          12.5831   7.647484e+15   

   mse_top100_periphery  pearson_distance_top100_periphery  \
0                0.3028                              0.076   

   edistance_top100_periphery  wasserstein_top100_periphery  \
0                      8.5412                      119.1635   

   sym_kldiv_top100_periphery  common_degs_top100_periphery  
0                5.958175e+15                             1  

  --- Held-out slide: GSM5808057 ---
Shapes for vstack -> Ctrl:(3455, 1053), Stim:(262, 1053), Pred:

Saved evaluation at: d:/Codes/0513/evaluation_results/cv/GSM5808057/KP
   mse_all_KP  pearson_distance_all_KP  edistance_all_KP  wasserstein_all_KP  \
0      0.2286                   0.0916           15.5137             15.5137   

   sym_kldiv_all_KP  mse_all  pearson_distance_all  edistance_all  \
0      7.588182e+15   0.2286                0.0916        15.5137   

   wasserstein_all  sym_kldiv_all  mse_top100_KP  pearson_distance_top100_KP  \
0          15.5137   7.588182e+15         0.4225                      0.0844   

   edistance_top100_KP  wasserstein_top100_KP  sym_kldiv_top100_KP  \
0               9.7097               134.0742         6.703710e+15   

   common_degs_top100_KP  
0                     38  
Shapes for vstack -> Ctrl:(3455, 1053), Stim:(87, 1053), Pred:(87, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/cv/GSM5808057/Tgfbr2
   mse_all_Tgfbr2  pearson_distance_all_Tgfbr2  edistance_all_Tgfbr2  \
0          0.3521                         0.13                19.255   

   wasserstein_all_Tgfbr2  sym_kldiv_all_Tgfbr2  mse_all  \
0                  19.255          8.226700e+15   0.3521   

   pearson_distance_all  edistance_all  wasserstein_all  sym_kldiv_all  \
0                  0.13         19.255           19.255   8.226700e+15   

   mse_top100_Tgfbr2  pearson_distance_top100_Tgfbr2  edistance_top100_Tgfbr2  \
0             0.3164                          0.0875                   9.1127   

   wasserstein_top100_Tgfbr2  sym_kldiv_top100_Tgfbr2  \
0                   134.5257             6.726282e+15   

   common_degs_top100_Tgfbr2  
0                         49  
Shapes for vstack -> Ctrl:(3455, 1053), Stim:(104, 1053), Pred:(104, 1053)


Saved evaluation at: d:/Codes/0513/evaluation_results/cv/GSM5808057/periphery
   mse_all_periphery  pearson_distance_all_periphery  edistance_all_periphery  \
0             0.1999                          0.0874                  14.5086   

   wasserstein_all_periphery  sym_kldiv_all_periphery  mse_all  \
0                    14.5086             7.832822e+15   0.1999   

   pearson_distance_all  edistance_all  wasserstein_all  sym_kldiv_all  \
0                0.0874        14.5086          14.5086   7.832822e+15   

   mse_top100_periphery  pearson_distance_top100_periphery  \
0                0.3444                             0.0673   

   edistance_top100_periphery  wasserstein_top100_periphery  \
0                      8.9972                      123.0849   

   sym_kldiv_top100_periphery  common_degs_top100_periphery  
0                6.154242e+15                             1  

CROSS-VALIDATION RESULTS
held_out_slide condition  mse_all_Jak2  pearson_distance_all_Jak2  edistanc